In [3]:
from __future__ import annotations
import time

class BaseDispatcher:
    def __init__(self):
        self.init_orders = []
        self.haul_orders = []
        self.back_orders = []
        self.init_order_count = 0
        self.haul_order_count = 0
        self.back_order_count = 0
        self.init_order_time = 0
        self.haul_order_time = 0
        self.back_order_time = 0
        self.total_order_count = 0
        self.total_order_time = 0

    def __getattribute__(self, name):
        attr = object.__getattribute__(self, name)
        if name in ["give_init_order", "give_haul_order", "give_back_order"] and callable(attr):
            return self._track_calls_and_time_wrapper(attr, name)
        return attr

    def update_mine(self, mine: "Mine"):
        # loadsite update
        for load_site in mine.load_sites:
            load_site.update_service_time()
            load_site.parking_lot.update_queue_wait_status()
        for dump_site in mine.dump_sites:
            dump_site.update_service_time()
            dump_site.parking_lot.update_queue_wait_status()
        # road update
        mine.update_road_status()

    def _track_calls_and_time_wrapper(self, method, method_type):
        def wrapper(*args, **kwargs):
            # check input
            # Handle implicit self parameter
            if not args and not kwargs:
                raise ValueError(f"{method_type} requires truck and mine arguments")

            # Pull truck and mine from positional or keyword arguments
            truck = None
            mine = None

            # Inspect keyword arguments
            if 'truck' in kwargs:
                truck = kwargs['truck']
            if 'mine' in kwargs:
                mine = kwargs['mine']

            # Fallback to positional arguments if missing
            if len(args) >= 2:  # self + truck + mine
                if truck is None:
                    truck = args[0]
                if mine is None:
                    mine = args[1]

            # Ensure both truck and mine are available
            if truck is None or mine is None:
                raise ValueError(f"{method_type} requires both truck and mine arguments")

            # update mine queue&wait info before the order starts
            # Refresh environment state via the mine object
            self.update_mine(mine)

            # Track dispatcher call count and execution time
            start_time = time.time()
            result = method(*args, **kwargs)
            elapsed_time = (time.time() - start_time) * 1000

            if method_type == "give_init_order":
                self.init_order_count += 1
                self.init_order_time += elapsed_time
                self.init_orders.append(int(result))
            elif method_type == "give_haul_order":
                self.haul_order_count += 1
                self.haul_order_time += elapsed_time
                self.haul_orders.append(int(result))
            elif method_type == "give_back_order":
                self.back_order_count += 1
                self.back_order_time += elapsed_time
                self.back_orders.append(int(result))

            self.total_order_count = self.init_order_count + self.haul_order_count + self.back_order_count
            self.total_order_time = self.init_order_time + self.haul_order_time + self.back_order_time

            return result
        return wrapper

    def give_init_order(self, truck: "Truck", mine: "Mine") -> int:
        raise NotImplementedError("Subclass must implement this method.")

    def give_haul_order(self, truck: "Truck", mine: "Mine") -> int:
        raise NotImplementedError("Subclass must implement this method.")

    def give_back_order(self, truck: "Truck", mine: "Mine") -> int:
        raise NotImplementedError("Subclass must implement this method.")


In [6]:
# public BaseDispatcher variable_1 = new BaseDispatcher()

variable_1 = BaseDispatcher()

In [9]:
variable_1.update_mine()

TypeError: BaseDispatcher.update_mine() missing 1 required positional argument: 'mine'